## 1. Importación de librerías

Framework: Pytorch

In [ ]:
!pip install opencv-python
!pip install -q torchinfo
!pip install -q ultralytics

In [2]:
# Librerías estándar de Python
import os
import sys
import json
import glob
import math
import copy
import random
import warnings
import time
import re
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import List, Tuple
from collections import OrderedDict, defaultdict
from math import sqrt

# Manejo de datos
import numpy as np
import pandas as pd

# Procesamiento de imágenes
import cv2
from PIL import Image

# Visualización y barras de Progreso
import matplotlib.pyplot as plt
from tqdm import tqdm

# Deep Learning: PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler, random_split

# Deep Learning: Torchvision (Computer Vision)
import torchvision
from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.ops import generalized_box_iou

# Librería Ultralitycs YOLO
from ultralytics import YOLO

%matplotlib inline
warnings.filterwarnings("ignore")

## 2. Descarga del dataset

3001 imágenes anotadas

| datos | nº imágenes | % |
| --- | --- | --- |
| training y validation | 2552 | 85,04% |
| test | 449 | 14,96% |

Estructura de directorios del entorno

In [ ]:
%%bash
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/IQBwS9aoohFwTLTjjUEdtw7gAYCxn3HUX7rWw0PaKVVNW8M?e=Ny6uQm&download=1" -O "crowdhuman.zip"
mkdir -p data/crowdhuman
unzip -q "crowdhuman.zip" -d data/crowdhuman
cp -r data/crowdhuman/valid/images/* data/crowdhuman/train/images/
cp -r data/crowdhuman/valid/labels/* data/crowdhuman/train/labels/

In [3]:
%%bash
echo "Entrenamiento y validación:" $(ls data/crowdhuman/train/images | wc -l) "imágenes"
echo "Test:" $(ls data/crowdhuman/test/images | wc -l) "imágenes"

Entrenamiento y validación: 2552 imágenes
Test: 449 imágenes


In [4]:
train_root = os.path.join("data", "crowdhuman", "train")
test_root = os.path.join("data", "crowdhuman", "test")
test_images_root = os.path.join("data", "crowdhuman", "test", "images")
train_images_root = os.path.join("data", "crowdhuman", "train", "images")


# Comprobación de dimensiones de las imágenes
for i, file in enumerate(sorted(os.listdir(train_images_root))[:10]):
    path = os.path.join(train_images_root, file)
    with Image.open(path) as img:
        print(f"{file}: {img.size}  (ancho x alto)")


273271-1017c000ac1360b7_jpg.rf.d03889e0dea9e6133159df81b602053a.jpg: (6567, 3888)  (ancho x alto)
273271-10355000e3a458a6_jpg.rf.2820608fdd399625c0253734e097ab9f.jpg: (600, 400)  (ancho x alto)
273271-1039400091556057_jpg.rf.479efc59e2b6f264cd91d0c958555350.jpg: (1993, 2274)  (ancho x alto)
273271-104ec00067d5b782_jpg.rf.65a6db8b2c97a9d0b1b7626eb40be668.jpg: (1024, 680)  (ancho x alto)
273271-1050b000e40d8e93_jpg.rf.779428c7f0638bea1e63b4040769d7e5.jpg: (700, 700)  (ancho x alto)
273271-105b40008a7b8b1f_jpg.rf.5ffa77a14893a6c8feef770f391ddaaa.jpg: (1600, 1067)  (ancho x alto)
273271-106220007be3af91_jpg.rf.c56b09a7ff6ef1e6273f4d0c3937ac2d.jpg: (1181, 800)  (ancho x alto)
273271-106f1000376a0c4c_jpg.rf.d8cae3fb877f00519c1dfdcdbebaaaec.jpg: (5184, 3456)  (ancho x alto)
273271-109db000477d4466_jpg.rf.454b254fd302cd6b4b7aae2219890c52.jpg: (1400, 787)  (ancho x alto)
273271-10a0a000dc884dfc_jpg.rf.c0024c195eec424f4ef9e8a34e45f83b.jpg: (1024, 685)  (ancho x alto)


Carga de las anotaciones desde archivos de texto con el formato YOLO

In [5]:
def cargar_anotaciones_cabezas(directorio_raiz):
    """
    Lee un dataset tipo YOLO, filtra solo la clase 1 (cabezas) y devuelve un diccionario de anotaciones tipo YOLO

    Args:
        directorio_raiz (str): Ruta a la carpeta que contiene las imágenes y 'labels'

    Devuelve:
        anotaciones (dict): Diccionario de anotaciones en el formato YOLO.
          Estructura:
                - Key: Nombre del archivo de imagen
                - Value: Lista de listas [[class, cx, cy, w, h], ...]
    """

    path_images = os.path.join(directorio_raiz, 'images')
    path_labels = os.path.join(directorio_raiz, 'labels')

    anotaciones = {}

    # Verificación de que los directorios existen
    if not os.path.exists(path_labels) or not os.path.exists(path_images):
        print("Error: No se encuentran las carpetas 'images' o 'labels'")
        return {}

    # Listamos todos los archivos .txt en el directorio de 'labels'
    archivos_txt = [f for f in os.listdir(path_labels) if f.endswith('.txt')]

    # Se lee cada archivo .txt asociado a una imagen del dataset
    for archivo_txt in archivos_txt:
        ruta_txt = os.path.join(path_labels, archivo_txt)

        yolo_labels = []

        with open(ruta_txt, 'r') as f:
            lineas = f.readlines()

            # Para cada línea, si la clase es 1 (cabeza), guardamos los datos de la anotación
            for linea in lineas:
                datos = linea.strip().split()
                if not datos: continue # Saltar líneas vacías

                clase = int(datos[0])

                if clase == 1:
                    cx = float(datos[1])
                    cy = float(datos[2])
                    w = float(datos[3])
                    h = float(datos[4])

                    # Guardamos en yolo_labels: [class, cx, cy, w, h]
                    yolo_labels.append([clase, cx, cy, w, h])

        # Si la imagen tiene al menos una anotación:
        if yolo_labels:
            nombre_imagen = os.path.splitext(archivo_txt)[0] + ".jpg" # Porque la imagen tiene el mismo nombre que el txt.

            # Guardamos todas las anotaciones relativas a la imagen
            if os.path.exists(os.path.join(path_images, nombre_imagen)):
                anotaciones[nombre_imagen] = yolo_labels

    return anotaciones

In [6]:
anotaciones = cargar_anotaciones_cabezas(train_root)

#Comprobación de las anotaciones
for id, (img_id, labels) in enumerate(anotaciones.items()):
        if id == 10:
          break
        labels_str = ",  ".join(f"clase: {clase} > ({cx},{cy},{bbox_w},{bbox_h})" for clase, cx, cy, bbox_w, bbox_h in labels)
        print(f"[{img_id}] -> [{labels_str}]")

[273271-1017c000ac1360b7_jpg.rf.d03889e0dea9e6133159df81b602053a.jpg] -> [clase: 1 > (0.04476930105070809,0.4965277777777778,0.05512410537536166,0.10262345679012345),  clase: 1 > (0.15174356631643063,0.5127314814814815,0.056799147251408555,0.1257716049382716),  clase: 1 > (0.2559768539668037,0.45949074074074076,0.05603776458047815,0.12988683127572018),  clase: 1 > (0.34719049794426676,0.5194187242798354,0.05512410537536166,0.11085390946502058),  clase: 1 > (0.4284300289325415,0.4488168724279835,0.048576214405360134,0.08076131687242798),  clase: 1 > (0.5185015989036089,0.5061728395061729,0.05177402162326786,0.1286008230452675),  clase: 1 > (0.6349170092888686,0.46566358024691357,0.048880767473732295,0.12834362139917696),  clase: 1 > (0.7271204507385411,0.4531893004115226,0.05299223389675651,0.08384773662551441),  clase: 1 > (0.8126998629511192,0.5042438271604939,0.05756052992233897,0.1257716049382716),  clase: 1 > (0.9402314603319628,0.4774948559670782,0.05162174508908177,0.093878600823

## 3. Preprocesamiento

**Reproducibilidad experimental**

El objetivo de la función *set_seed* es garantizar resultados consistentes utilizando los mismos datos y parámetros, reducir la variabilidad no explicada entre distintas ejecuciones de entrenamiento. Si se cambia un parámetro (p.ej. learning rate) de un entrenamiento respecto de otro debemos ser capaces de explicar el efecto de este sobre los resultados.

In [7]:
#Establecer semilla para que las ejecuciones sean reproducibles
def set_seed(seed):
    random.seed(seed)           #Módulos estándar de Python
    np.random.seed(seed)        #Operaciones de NumPy
    torch.manual_seed(seed)     #CPU y GPU por defecto

    if torch.cuda.is_available():
       torch.cuda.manual_seed(seed)
       torch.cuda.manual_seed_all(seed)    #Todas las GPUs en el sistema

       #Modo de búsqueda donde cuDNN prueba múltiples algoritmos para una operación dada y selecciona el más rápido para el tamaño de entrada específico
       torch.backends.cudnn.benchmark = False #False = Evita la selección dinámica de algoritmos
       #Obliga a la biblioteca a utilizar únicamente algoritmos que son deterministas
       torch.backends.cudnn.deterministic = True

**Configuración de parámetros de entrenamiento**

In [9]:
VARIANTES_YOLO = {
    "yolov8n": dict(width=0.25, depth=0.33, max_ch=1024, pt="yolov8n.pt"),
    "yolov8s": dict(width=0.50, depth=0.33, max_ch=1024, pt="yolov8s.pt"),
    "yolov8m": dict(width=0.75, depth=0.67, max_ch=768,  pt="yolov8m.pt"),
    "yolov8l": dict(width=1.00, depth=1.00, max_ch=512,  pt="yolov8l.pt"),
    "yolov8x": dict(width=1.25, depth=1.00, max_ch=512,  pt="yolov8x.pt"),
}

@dataclass(frozen=True)   # Constructor, fuertemente tipado, frozen=True (instancia inmutable)
class TrainingConfig:
      img_size: int = 640             #Resolución de entrada de la imagen en el modelo
      num_clases: int = 1             #Número de categorías de objetos (solo persona)
      batch_size: int = 16            #Muestras de entrenamiento procesadas simultaneamente
      num_epochs: int = 50           #Número de iteraciones completas sobre el conjunto de datos

      learning_rate: float = 1.0e-4   #Tasa de aprendizaje
      weight_decay: float = 5e-5      #Coeficiente de regularización L2
      warmup_epochs: int = 3          #Número de épocas iniciales para estabilización
      warmup_start: float = 0.1       #Factor multiplicador inicial del lr durante el warmup

      conf_thres: float = 0.25        #Umbral de confianza
      iou_thres: float = 0.45         #Umbral de intersección (para NMS)
      device: str = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Hardware de cómputo

      # Parámetro de variante
      variante: str = "yolov8n"

      @property
      def width(self):   return VARIANTES_YOLO[self.variante]["width"]  #Factor de ancho del modelo (número de canales por capa)
      @property
      def depth(self):   return VARIANTES_YOLO[self.variante]["depth"]  #Factor de profundidad (número de repeticiones de bloques)
      @property
      def max_ch(self):  return VARIANTES_YOLO[self.variante]["max_ch"]
      @property
      def pt_file(self): return VARIANTES_YOLO[self.variante]["pt"]     #Pesos preentrenados

train_config = TrainingConfig()

In [10]:
def resize_image(img: Image.Image, new_size=train_config.img_size, color=(114,114,114)):
    """
    Función de redimensionado que guarda la relación de aspecto de la imagen.

    Args:
        - img (Image.Image): Imagen original en formato PIL
        - new_size (int): Tamaño de la imagen de salida
        - color (tuple): Valor RGB de relleno de zonas 'vacías'

    Devuelve:
        - img_padded: Imagen redimensionada
        - r: Ratio aplicado
        - pad_left, pad_top: Desplazamiento en píxeles aplicado para centrar la imagen
        - new_w, new_h: Dimensiones nuevas
        - w0, h0: Dimensiones originales
    """
    w0, h0 = img.size
    r = min(new_size / w0, new_size / h0)
    new_w = int(round(w0 * r))
    new_h = int(round(h0 * r))

    img_resized = img.resize((new_w, new_h), resample=Image.BILINEAR)

    pad_w = new_size - new_w
    pad_h = new_size - new_h
    pad_left = pad_w // 2
    pad_top = pad_h // 2

    canvas = Image.new("RGB", (new_size, new_size), color) # crear canvas de fondo
    canvas.paste(img_resized, (pad_left, pad_top))         # situar la imagen en el canvas

    return canvas, r, pad_left, pad_top, new_w, new_h, w0, h0


class YOLODataset(Dataset):
    """
    Clase heredada de torch.utils.data.Dataset, responsable de sincronizar la transformación de la imagen con la transformación de sus anotaciones (boxes)
    """
    def __init__(self, images_dir, annotations_dict, img_size):
        self.images_dir = Path(images_dir)
        self.annotations = annotations_dict
        self.file_list = list(annotations_dict.keys())
        self.img_size = train_config.img_size

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_name = self.file_list[idx]
        img_path = self.images_dir / img_name

        img = Image.open(img_path).convert("RGB")
        img_lb, r, pad_left, pad_top, new_w, new_h, w0, h0 = resize_image(img, self.img_size)

        img_t = TF.to_tensor(img_lb)        # Transformación a tensor [0,1]
        boxes = self.annotations[img_name]  # lista de [cls,cx,cy,w,h] originales

        #Si no hay anotaciones, devolvemos tensor vacío
        if len(boxes) == 0:
            target = torch.zeros((0, 5), dtype=torch.float32)
            return img_t, target

        boxes = torch.tensor(boxes, dtype=torch.float32)  # [N,5]
        cls = boxes[:, 0:1] #clase

        # Des-normalización. Pasar a pixeles absolutos
        cx = boxes[:, 1] * w0
        cy = boxes[:, 2] * h0
        bw = boxes[:, 3] * w0
        bh = boxes[:, 4] * h0

        # De (cx,cy,w,h) a esquinas (x1,y1,x2,y2)
        x1 = cx - bw / 2
        y1 = cy - bh / 2
        x2 = cx + bw / 2
        y2 = cy + bh / 2

        # Transformación geométrica (misma escala y desplazamiento que se aplicó a la imagen)
        x1 = x1 * r + pad_left
        x2 = x2 * r + pad_left
        y1 = y1 * r + pad_top
        y2 = y2 * r + pad_top

        # clip de seguridad, que ninguna coordenada se salga de las dimensiones debido a errores de redondep
        x1 = x1.clamp(0, self.img_size)
        y1 = y1.clamp(0, self.img_size)
        x2 = x2.clamp(0, self.img_size)
        y2 = y2.clamp(0, self.img_size)

        # Vuelta a la normalización en formato YOLO (cx,cy,w,h)
        cx2 = (x1 + x2) / 2 / self.img_size
        cy2 = (y1 + y2) / 2 / self.img_size
        bw2 = (x2 - x1) / self.img_size
        bh2 = (y2 - y1) / self.img_size

        target = torch.cat([cls, cx2.unsqueeze(1), cy2.unsqueeze(1), bw2.unsqueeze(1), bh2.unsqueeze(1)], dim=1) # unsqueeze(1) ajuste de dimensiones [N, 1]
        return img_t, target

## 4. Construcción del modelo

Funciones de bloques del modelo

In [11]:
def autopad(k, p=None, d=1):
    """
    Cálculo de padding para garantizar que la dimensión de salida sea idéntica a la de entrada cuando el stride es 1
    """
    if d > 1: k = d * (k - 1) + 1 if isinstance(k, int) else [d * (x - 1) + 1 for x in k]
    if p is None: p = k // 2 if isinstance(k, int) else [x // 2 for x in k]
    return p

###### Bloque Convolucional Base ######
class Conv(nn.Module):
    """
    Unidad atómica de la red: Convolución + Normalización + Activación
      c1 = canales de entrada         p = padding
      c2 = caneles de salida          g = grupos (convolución en grupos)
      k = tamaño del kernel           d = dilatación
      s = stride                      act = función de activación SiLU
    """
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU() if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

###### Bottleneck (ResNet) ######
class Bottleneck(nn.Module):
    """
    Bloque residual clásico de ResNet. Aumenta la profundidad de la red sin que resulte en desvanecimiento del gradiente
      e = canales de la capa intermedia (expansión)
      shortcut = se decide si sumar la entrada x al resultado
    """
    def __init__(self, c1, c2, shortcut=True, g=1, k=(3, 3), e=0.5):
        super().__init__()
        c = int(c2 * e)
        self.cv1 = Conv(c1, c, k[0], 1)
        self.cv2 = Conv(c, c2, k[1], 1, g=g)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))

###### C2f ######
class C2f(nn.Module):
    """
    CSP Bottleneck con 2 convoluciones. Evolución del bloque C3 que mejora el flujo de gradientes mediante múltiples caminos de ramificación
      n = profundidad del bloque
    """
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * c, 1, 1)
        self.cv2 = Conv((2 + n) * c, c2, 1)
        self.m = nn.ModuleList(Bottleneck(c, c, shortcut, g, e=1.0) for _ in range(n))

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        for m in self.m:
            y.append(m(y[-1]))
        return self.cv2(torch.cat(y, 1))

###### SPPF ######
class SPPF(nn.Module):
    """
    Spatial Pyramid Pooling - Fast. Módulo que permite a la red ver el objeto a múltiples escalas. Hace tres poolings de 5x5 en serie
      k = tamaño del kernel del pooling
    """
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c = c1 // 2
        self.cv1 = Conv(c1, c, 1, 1)
        self.cv2 = Conv(c * 4, c2, 1, 1)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        return self.cv2(torch.cat((x, y1, y2, self.m(y2)), 1))

###### Decoupled Head (Cabecera Desacoplada) ######
class DetectHead(nn.Module):
    """
    Convierte los mapas de características en predicciones concretas (box, presencia de objeto y clase)
      nc = número de clases
      ch = canales de entrada
    """
    def __init__(self, nc, ch=()):
        super().__init__()
        self.nc = train_config.num_clases
        self.nl = len(ch)

        self.box_convs = nn.ModuleList(nn.Sequential(Conv(x, x, 3), Conv(x, x, 3), nn.Conv2d(x, 4, 1)) for x in ch)   #capa de regresión de cajas, 4 = (x, y, w, h)
        self.obj_convs = nn.ModuleList(nn.Sequential(Conv(x, x, 3), Conv(x, x, 3), nn.Conv2d(x, 1, 1)) for x in ch)   #capa de objectness
        self.cls_convs = nn.ModuleList(nn.Sequential(Conv(x, x, 3), Conv(x, x, 3), nn.Conv2d(x, nc, 1)) for x in ch)  #capa de clases

        #Se inicia el sesgo de la capa de 'objeto' para que prediga una probabilidad baja (p=0.01) y evitar que el error inicial se dispare
        for m in self.obj_convs:
            conv = m[-1]  # último Conv2d de obj_convs
            if isinstance(conv, nn.Conv2d) and conv.bias is not None:
                conv.bias.data.fill_(-math.log((1 - 0.01) / 0.01))  # fórmula del bias

        for m in self.cls_convs:
            conv = m[-1]  # último Conv2d
            if isinstance(conv, nn.Conv2d) and conv.bias is not None:
                conv.bias.data.zero_()

    def forward(self, x):
        """
        x = lista [P3, P4, P5], las tres escalas de detección
        outputs = lista de tensores [B, C, H, W] donde C = (x, y, w, h, objectness, nc)
        """
        outputs = []
        for i in range(self.nl):
            box_out = self.box_convs[i](x[i])
            obj_out = self.obj_convs[i](x[i])
            cls_out = self.cls_convs[i](x[i])
            outputs.append(torch.cat([box_out, obj_out, cls_out], 1))
        return outputs

Funciones de escalamiento que permiten que el mismo código pueda modificarsea un modelo Nano, Small, Medium o Large

In [12]:
#Escalador de ancho de canales
def ch(c, w, max_ch=1024):
    return min(int(c * w), max_ch)

#Escalador de profundidad (cuántas veces se repite un bloque C2f)
def d(n, depth):
    return max(1, round(n * depth))

Clase principal del modelo que ensambla todos los bloques

In [13]:
class YOLO_model(nn.Module):
    """
    Arquitectura YOLO (Backbone + Neck + Head)
      Recibe tensor de imágenes normalizado: [Batch, 3, H, W]
      Devuelve lista de 3 tensores de predicción para cada escala: [[B, 6, H/8, W/8], [B, 6, H/16, W/16], [B, 6, H/32, W/32]]
        donde 6 = (x, y, w, h, objectness, nc)
    """
    def __init__(self, num_classes=1, width=train_config.width, depth=train_config.depth, max_ch=train_config.max_ch):
        super().__init__()

        def c(base): return ch(base, width, max_ch)

        # BACKBONE
        self.stem = Conv(3, c(64), 3, 2)
        self.conv2 = Conv(c(64), c(128), 3, 2)
        self.c2f_2 = C2f(c(128), c(128), n=d(3, depth))

        self.conv3 = Conv(c(128), c(256), 3, 2)
        self.c2f_3 = C2f(c(256), c(256), n=d(6, depth))

        self.conv4 = Conv(c(256), c(512), 3, 2)
        self.c2f_4 = C2f(c(512), c(512), n=d(6, depth))

        self.conv5 = Conv(c(512), c(1024), 3, 2)
        self.c2f_5 = C2f(c(1024), c(1024), n=d(3, depth))
        self.sppf = SPPF(c(1024), c(1024), k=5)

        # NECK (PANet simple)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')

        # P5 -> P4
        self.reduce_p5 = Conv(c(1024), c(512), 1, 1)
        self.c2f_p4 = C2f(c(512) + c(512), c(512), n=d(3, depth))

        # P4 -> P3
        self.reduce_p4 = Conv(c(512), c(256), 1, 1)
        self.c2f_p3 = C2f(c(256)+c(256), c(256), n=d(3, depth))

        # PAN Bottom-Up
        self.down_p3 = Conv(c(256), c(256), 3, 2)
        self.c2f_n4 = C2f(c(256) + c(512), c(512), n=d(3, depth))

        self.down_n4 = Conv(c(512), c(512), 3, 2)
        self.c2f_n5 = C2f(c(512) + c(1024), c(1024), n=d(3, depth))

        # HEAD
        self.head = DetectHead(nc=num_classes, ch=(c(256), c(512), c(1024)))

    def forward(self, x):
        # Backbone
        p1 = self.stem(x)
        p2 = self.c2f_2(self.conv2(p1))
        p3 = self.c2f_3(self.conv3(p2))
        p4 = self.c2f_4(self.conv4(p3))
        p5 = self.sppf(self.c2f_5(self.conv5(p4)))

        # Neck Top-Down
        p5_reduced = self.reduce_p5(p5)
        p4_cat = torch.cat([self.up(p5_reduced), p4], 1)
        p4_out = self.c2f_p4(p4_cat)

        p4_reduced = self.reduce_p4(p4_out)
        p3_cat = torch.cat([self.up(p4_reduced), p3], 1)
        p3_out = self.c2f_p3(p3_cat)

        # Neck Bottom-Up
        n4_in = torch.cat([self.down_p3(p3_out), p4_out], 1)
        n4_out = self.c2f_n4(n4_in)

        n5_in = torch.cat([self.down_n4(n4_out), p5], 1)
        n5_out = self.c2f_n5(n5_in)

        return self.head([p3_out, n4_out, n5_out])

## 5. Función de pérdida (YOLOLoss)

Funciones auxiliares, herramientas de cálculo geométrico

In [14]:
def xywh_to_xyxy(box):
    """
    Transformaciones lineales para cambiar de formato de centros (x, y, w, h) a esquinas (x1, y1, x2, y2) de los boxes
    """
    cx, cy, w, h = box.unbind(-1)
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    return torch.stack((x1, y1, x2, y2), dim=-1)

def decode_to_xywh(pred_box_raw, grid_xy, stride, img_size, device):
    """
    Decodifica prediciones 'brutas' o 'crudas' (raw) en boxes reales normalizados

    Args:
      - pred_box_raw: [...,4] 'logits' de predicción de la red
      - grid_xy: [...,2] (x, y) posición de la celda en el grid
      - stride: pixeles por celda del grid
      - img_size: tamaño de la imagen

    Devuelve:
      - boxes normalizadas [0,1] en formato (x, y, w, h)
    """

    #Se amplía el rango de predicción (-0.5, 1.5), convirte el offset relativo a la celda en posición dentro del grid (+ grid_xy), se pasa a pixeles
    xy = (pred_box_raw[..., 0:2].sigmoid() * 2.0 - 0.5 + grid_xy) * stride
    #Se amplía el rango (0, 2), se cuadra para evitar negativos, se pasa a pixeles
    wh = (pred_box_raw[..., 2:4].sigmoid() * 2.0) ** 2 * stride

    #Normalizar a [0, 1]
    xy = xy / img_size
    wh = wh / img_size
    return torch.cat([xy, wh], dim=-1)

def ciou_loss_single(preds_xywh, targets_xywh, eps=1e-7):
    """
    CIoU (Complete IoU): Miramos la superposición entre el area target y la predicción (IoU clásico), la distancia entre los centros y su forma (Aspect Ratio)

    Args:
      - preds_xywh, targets_xywh: (...,4), donde 4 es [x,y,w,h] (normalizado [0,1])
      - eps: valor pequeño para evitar la división por cero que provocaría que valores NaN se propagaran a través de la red

    Devuelve:
      CIoU (no la pérdida): el valor de CIoU entre (-inf, 1], donde 1 = perfecto
      Para pérdida se usa (1 - CIoU)
    """

    #Conversión de centros (XYWH) a esquinas (XYXY)
    pred_xyxy = xywh_to_xyxy(preds_xywh)
    tgt_xyxy = xywh_to_xyxy(targets_xywh)

    # Área de intersección entre predicción y target
    x1 = torch.max(pred_xyxy[..., 0], tgt_xyxy[..., 0])
    y1 = torch.max(pred_xyxy[..., 1], tgt_xyxy[..., 1])
    x2 = torch.min(pred_xyxy[..., 2], tgt_xyxy[..., 2])
    y2 = torch.min(pred_xyxy[..., 3], tgt_xyxy[..., 3])

    inter_w = (x2 - x1).clamp(min=0) #Si las boxes no se solapan, inter_w será negativo, por eso clamp(min=0)
    inter_h = (y2 - y1).clamp(min=0)
    inter_area = inter_w * inter_h

    # Área de unión
    area_pred = (pred_xyxy[..., 2] - pred_xyxy[..., 0]).clamp(min=eps) * (pred_xyxy[..., 3] - pred_xyxy[..., 1]).clamp(min=eps)
    area_tgt = (tgt_xyxy[..., 2] - tgt_xyxy[..., 0]).clamp(min=eps) * (tgt_xyxy[..., 3] - tgt_xyxy[..., 1]).clamp(min=eps)

    union = area_pred + area_tgt - inter_area + eps

    #Cálculo de IoU
    iou = inter_area / union

    #Cáculo de penalizaciones

    cw = (torch.max(pred_xyxy[..., 2], tgt_xyxy[..., 2]) - torch.min(pred_xyxy[..., 0], tgt_xyxy[..., 0])).clamp(min=eps) #Ancho envolvente convexo
    ch = (torch.max(pred_xyxy[..., 3], tgt_xyxy[..., 3]) - torch.min(pred_xyxy[..., 1], tgt_xyxy[..., 1])).clamp(min=eps) #Alto envolvente convexo
    c2 = (cw ** 2 + ch ** 2).clamp(min=eps) #Diagonal al cuadrado

    #Distancia de los centros
    rho2 = ((preds_xywh[..., 0] - targets_xywh[..., 0]) ** 2 + (preds_xywh[..., 1] - targets_xywh[..., 1]) ** 2)

    #Aspecto, forma
    w_pred = preds_xywh[..., 2].clamp(min=eps)
    h_pred = preds_xywh[..., 3].clamp(min=eps)
    w_t = targets_xywh[..., 2].clamp(min=eps)
    h_t = targets_xywh[..., 3].clamp(min=eps)
    v = (4 / (math.pi ** 2)) * torch.pow(torch.atan(w_t / h_t) - torch.atan(w_pred / h_pred), 2)

    with torch.no_grad():
        alpha = v / (1 - iou + v + eps)

    ciou = iou - (rho2 / c2 + alpha * v)
    return ciou.clamp(min=-1.0, max=1.0)  #para estabilizar el rango

**Función principal**

Los algoritmos YOLO modernos no optimizan una única métrica Loss, si no que resuelven simultáneamente tres problemas de optimización distintos. La función de pérdida agregada, $L_{total}$, es la suma ponderada de tres componentes:


*   Pérdida de regresión de box ($L_{box}$)
*   Pérdida de objeto/confianza ($L_{obj}$)
*   Pérdida de clasificación ($L_{cls}$)


In [15]:
class YOLOLoss(nn.Module):
    """
    - Loss de box: CIoU
    - Loss de obj: Focal BCE
    - Loss de cls: BCE (solo en positivos) para evitar que el background meta ruido
    - Asignación:
        Cada GT se asigna a una escala según su área
        Se asigna a vecindario 3x3 alrededor del centro para reducir colisiones
    """

    def __init__(self, strides=(8,16,32), img_size=train_config.img_size, box_weight=7.5, obj_weight=0.75, cls_weight=0.5, focal_gamma=2.0,
                 focal_alpha=0.25, center_radius=0):
        super().__init__()
        self.strides = strides
        self.img_size = img_size

        self.box_weight = box_weight
        self.obj_weight = obj_weight
        self.cls_weight = cls_weight

        self.focal_gamma = focal_gamma #parámetro de enfoque
        self.focal_alpha = focal_alpha
        self.center_radius = center_radius

    def focal_bce(self, logits, targets):
        """
        Focal BCE: Focal Loss binaria clásica, reduce el peso de ejemplos fáciles y enfatiza los difíciles
        focal = alpha*(1-pt)^gamma*BCE
        """
        #Función Pytorch que calcula la Binary Cross Entropy (BCE)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")

        #Convertir logits a probabilidades
        p = torch.sigmoid(logits)

        # pt = probabilidad que el modelo asigna a la clase correcta
        pt = p * targets + (1 - p) * (1 - targets)
        alpha = self.focal_alpha * targets + (1 - self.focal_alpha) * (1 - targets)

        focal = alpha * (1 - pt).pow(self.focal_gamma) * bce
        return focal

    def choose_scales(self, w_norm, h_norm):
        area = (w_norm * h_norm).item()
        if area < 0.008:
            return [0]          # P3
        elif area < 0.015:
            return [0, 1]       # P3 + P4
        elif area < 0.035:
            return [1]          # P4
        elif area < 0.06:
            return [1, 2]       # P4 + P5
        else:
            return [2]          # P5


    def forward(self, preds, targets):
        """
        Args:
          - preds: lista de tensores [B, 4 + nc, H, W] (en el orden de self.strides)
          - targets: tensor [N,6] (batch_idx, class, x, y, w, h) normalizados a img_size
        """
        device = preds[0].device

        #Inicialización de tensores
        total_box = torch.tensor(0.0, device=device)
        total_obj = torch.tensor(0.0, device=device)
        total_cls = torch.tensor(0.0, device=device)
        total_pos = 0

        if targets is None or targets.numel() == 0:
            #Batch sin GT. Para estabilidad se devuelve 0
            return torch.zeros((), device=device, requires_grad=True)

        #Asegurarse de que targets es un tensor
        targets = targets.to(device)

        #En este caso cls será 1 (cabezas), lo normalizaremos a 0 para entrenamiento de 1 clase
        targets = targets.clone()
        targets[:, 1] = 0.0

        #Iteración de las 3 escalas
        for si, pred in enumerate(preds):
            stride = self.strides[si]
            B, C, H, W = pred.shape
            nc = C - 5  # 4 box + 1 obj + nc

            pred = pred.permute(0, 2, 3, 1).contiguous()  # [B,H,W,C] Se necesita que los canales sean la última dimensión

            box_raw = pred[..., 0:4]
            obj_logit = pred[..., 4]
            cls_logits = pred[..., 5:5+nc]  # una clase

            #Creación del grid según el stride, cada celda es responsable de predecir un objeto si su centro cae ahí (grid [H,W,2])
            grid_y, grid_x = torch.meshgrid(
                torch.arange(H, device=device),
                torch.arange(W, device=device),
                indexing="ij"
            )
            grid = torch.stack([grid_x, grid_y], dim=-1).float() # grid pasa a [B,H,W,2]

            #Decodificar las boxes de la salida de la red a coordenadas normalizadas [0, 1] [B,H,W,4] con la función auxiliar
            pred_boxes = decode_to_xywh(box_raw, grid.unsqueeze(0), stride, self.img_size, device)

            #Preparar los mapas target del Ground Truth
            obj_t = torch.zeros((B, H, W), device=device)     #Mapa de presencia de objetos
            cls_t = torch.zeros((B, H, W, nc), device=device) #Mapa para guardar el tipo de clase (solo hay una)
            box_t = torch.zeros((B, H, W, 4), device=device)  #Mapa para guardar los boxes reales

            ###########
            #Asignación por batch
            #iteramos targets y solo asignamos los que pertenecen a esta escala
            for t in targets:
                b, cls, x, y, w, h = t
                b = int(b)

                # elegir escala por área
                scales = self.choose_scales(w, h)
                if si not in scales:
                    continue

                gx = x * W
                gy = y * H
                #(ix, iy) = coordenadas de la celda central del objeto
                ix = int(torch.clamp(gx.floor(), 0, W - 1).item())
                iy = int(torch.clamp(gy.floor(), 0, H - 1).item())

                #Expansión de vecindario
                for dy in range(-self.center_radius, self.center_radius + 1):
                    for dx in range(-self.center_radius, self.center_radius + 1):
                        jx = ix + dx
                        jy = iy + dy
                        if jx < 0 or jx >= W or jy < 0 or jy >= H:
                            continue
                        obj_t[b, jy, jx] = 1.0                                          #Meter en el mapa que en esa celda hay objeto
                        box_t[b, jy, jx] = torch.tensor([x, y, w, h], device=device)    #Meter en el mapa la caja del objeto
                        cls_t[b, jy, jx, int(cls.item())] = 1.0  #nc=1 => índice 0

            #Solo calculamos el error de posición (box) para las celdas que deberían tener un objeto
            pos = obj_t > 0.5
            n_pos = int(pos.sum().item())
            total_pos += n_pos

            ###########
            #1. Box loss: CIoU solo positivos
            if n_pos > 0:
                ciou = ciou_loss_single(pred_boxes[pos], box_t[pos]) #Función auxiliar cálculo de CIoU
                total_box += (1.0 - ciou).sum()

                #2. Cls loss: reduce falsos positivos en el background
                cls_loss = F.binary_cross_entropy_with_logits(cls_logits[pos], cls_t[pos], reduction="sum")
                total_cls += cls_loss

            #3. Obj loss: focal sobre todo, pero normalizado por n_pos para no diluir positivos
            obj_loss_map = self.focal_bce(obj_logit, obj_t)  # [B,H,W]
            pos_mask = obj_t > 0.5
            neg_mask = ~pos_mask

            #positivos: todos
            pos_loss = obj_loss_map[pos_mask]

            #negativos: quedarnos con top-k "hard" según loss (los k peores negativos)
            neg_loss = obj_loss_map[neg_mask]
            # regla, se supone que k es al menos 1000 o 10 veces el número de positivos
            k = min(neg_loss.numel(), max(1000, 10 * pos_loss.numel()))
            if k > 0:
                neg_loss, _ = torch.topk(neg_loss.flatten(), k)

            total_obj += pos_loss.sum() + neg_loss.sum()
            ###########

        # división entre d para que el loss sea invariante al tamaño del batch y cantidad de personas en la imagen
        d = max(total_pos, 1)
        # se pondera la agregación
        loss_box = self.box_weight * (total_box / d)
        loss_obj = self.obj_weight * (total_obj / d)
        loss_cls = self.cls_weight * (total_cls / d)

        return loss_box + loss_obj + loss_cls

## 6. Recogida de Logs

In [16]:
class logs_function:

    def __init__(self, output_dir: str, num_epochs: int):
        self.output_dir = output_dir
        self.num_epochs = num_epochs

        os.makedirs(self.output_dir, exist_ok=True)

        log_files_count = len(glob.glob(os.path.join(self.output_dir, "*.txt")))
        self.output_file_dir = os.path.join(
            self.output_dir,
            f"Log_{log_files_count}_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.txt"
        )

        self.create_log_file()

    def create_log_file(self):
        with open(self.output_file_dir, 'w') as wr:
            wr.write(f"Fecha de creación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            wr.flush()

    def log_hyperparams(self, **config):
        with open(self.output_file_dir, 'a') as wr:
            wr.write("\n======= Configuración del entrenamiento =======\n")
            for key, value in config.items():
                wr.write(f"- {key}: {value}\n")
            wr.write("================================================\n\n")
            wr.flush()

    def log_results(self, epoch, **metrics):
        with open(self.output_file_dir, 'a') as writer:
            log = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] [{epoch}/{self.num_epochs}] "
            for key, value in metrics.items():
                if isinstance(value, (int, float)):
                    log += f"| {key}: {round(value, 5)} "
                else:
                    log += f"| {key}: {value} "

            writer.write(log + "\n")
            writer.flush()

    def write(self, text):
        with open(self.output_file_dir, 'a') as wr:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            wr.write(f"[{timestamp}] {text}\n")
            wr.flush()
        print(text)

    def log_error(self, err):
        with open(self.output_file_dir, 'a') as wr:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            wr.write(f"[{timestamp}] ERROR: {err}\n")
            wr.flush()
        print("ERROR:", err)

## 7. Entrenamiento del modelo

Funciones de inferencia y visualización del progreso durante el entrenamiento

In [17]:
def postprocess_single_prediction(prediction, img_size=train_config.img_size, conf_thres=0.25, iou_thres=0.65, max_det=500, return_scores=False):
    """
    Decodifica la salida raw de la red neuronal a coordenadas de imagen (xyxy),
    filtra por confianza y aplica Non-Maximum Suppression (NMS).

    Args:
      - prediction (list[Tensor]): Lista de 3 tensores [1, C, H, W] (escalas P3, P4, P5)
                                   C debe ser (4 coords + 1 obj + 1 cls)
                                   Solo soporta batch_size = 1
      - conf_thres: Umbral mínimo de confianza (obj * cls) para considerar una detección
      - iou_thres: Umbral de IoU para el NMS (eliminar cajas duplicadas)
      - max_det: Número máximo de detecciones permitidas por imagen
      - return_scores: True, devuelve tensores [N, 6] (x1,y1,x2,y2,score,cls)
                        False, devuelve solo [N, 4] (x1,y1,x2,y2)

    Devuelve:
        Tensor de boxes o None si no hay detecciones
    """
    if not isinstance(prediction, (list, tuple)):
        prediction = [prediction]

    device = prediction[0].device
    strides = [8, 16, 32]

    all_boxes = []
    all_scores = []

    for i, pred in enumerate(prediction):
        stride = strides[i]
        B, C, H, W = pred.shape

        # [B, C, H, W] -> [N, C]
        pred = pred.view(B, C, -1).permute(0, 2, 1).squeeze(0)
        if C < 6:
            raise ValueError("Se esperan al menos 6 canales (xywh + obj + cls)")

        #Separar componentes
        box_raw = pred[:, 0:4]
        obj = pred[:, 4].sigmoid()
        cls = pred[:, 5].sigmoid()
        score = obj * cls

        #Generar el grid
        yv, xv = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing="ij")
        grid = torch.stack((xv, yv), 2).view(-1, 2).float()

        #Decodificar: raw a xywh normalizado [0,1]
        boxes_norm = decode_to_xywh(box_raw, grid, stride, img_size, device)
        #Desnormalizar: [0,1] a [0, img_size] (pixeles)
        boxes_pixel = boxes_norm * img_size
        #Transformar: centro (xywh) a esquinas (xyxy)
        boxes_xyxy = xywh_to_xyxy(boxes_pixel)

        #Filtrado por confianza
        mask = score > conf_thres
        if not mask.any():
            continue

        boxes = boxes_xyxy[mask]
        scores = score[mask]

        #Clip de seguridad para no salirse de la imagen
        boxes[:, 0].clamp_(0, img_size)
        boxes[:, 1].clamp_(0, img_size)
        boxes[:, 2].clamp_(0, img_size)
        boxes[:, 3].clamp_(0, img_size)

        all_boxes.append(boxes)
        all_scores.append(scores)

    if len(all_boxes) == 0:
        return None
    #Concatenar todos los strides
    boxes = torch.cat(all_boxes, dim=0)
    scores = torch.cat(all_scores, dim=0)

    #NMS global
    keep = torchvision.ops.nms(boxes, scores, iou_thres)
    keep = keep[:max_det]

    boxes = boxes[keep]
    scores = scores[keep]

    if return_scores:
        classes = torch.zeros_like(scores) #Todo clase 0
        return torch.cat([boxes, scores.unsqueeze(1), classes.unsqueeze(1)], dim=1)

    return boxes

def plot_progress(model, sample_img_t, sample_targets_norm, epoch, fold, device, save_dir):
    """
    Genera y guarda una imagen comparativa entre el Ground Truth y predicción
    Args:
      - sample_img_t: Tensor [1, 3, H, W] normalizado.
      - sample_targets_norm: Tensor [N, 5] (cls, xn, yn, wn, hn) normalizado.
    """
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    img_size = sample_img_t.shape[2]

    #Preparar la imagen base de fondo donde se van a colocar las otras
    img_np = sample_img_t.squeeze(0).permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * 255).astype(np.uint8)
    img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    img_gt = img_np.copy()
    img_pred = img_np.copy()

    #Dibujar GT (Izquierda)
    gt_count = 0
    if sample_targets_norm is not None:
        #Desnormalizar targets de (x, y, w, h) a píxeles
        boxes_gt = sample_targets_norm[:, 1:] * img_size
        for box in boxes_gt:
            gt_count += 1
            xc, yc, w, h = box
            x1, y1 = int(xc - w/2), int(yc - h/2)
            x2, y2 = int(xc + w/2), int(yc + h/2)
            cv2.rectangle(img_gt, (x1, y1), (x2, y2), (0, 255, 0), 2) #color verde

    #Obtener y dibujar predicciones (Derecha)
    pred_count = 0
    with torch.no_grad():
        preds_raw = model(sample_img_t.to(device))
        #Post-procesado de la imagen
        final_boxes = postprocess_single_prediction(preds_raw, img_size, conf_thres=train_config.conf_thres)

    if final_boxes is not None:
        for box in final_boxes:
            pred_count += 1
            x1, y1, x2, y2 = box.int().cpu().numpy()
            cv2.rectangle(img_pred, (x1, y1), (x2, y2), (0, 0, 255), 2) #color rojo

    #Crear la figura Matplotlib
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    #GT plot
    axes[0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Ground Truth (Count: {gt_count})")
    axes[0].axis('off')

    #Pred plot
    axes[1].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Predicción en Epoch {epoch} (Count: {pred_count})")
    axes[1].axis('off')

    plt.suptitle(f"Progreso Fold {fold} - Epoch {epoch}")
    save_path = os.path.join(save_dir, f"fold{fold}_epoch{epoch:03d}.png")
    plt.savefig(save_path)
    plt.close(fig) # Cerrar para liberar memoria

**Funciones Transfer Learning**

Conjunto de funciones para cargar pesos preentrenados YOLO de Ultralytics y utilizarlos en la implementación propia de YOLO mediante el manejo de las diferencias de nombres y arquitectura del modelo.

In [18]:
def get_ultralytics_sd(pt_path="yolov8n.pt", device="cpu"):
    """
    Carga un modelo de la librería Ultralytics y extrae sus pesos (state_dict)
    y limpia los prefijos originales, lo que facilita el mapeo
    """
    y = YOLO(pt_path)
    m = y.model.to(device)
    sd = OrderedDict() #uso de un diccionario ordenado
    for k, v in m.state_dict().items():
        nk = k[len("model."):] if k.startswith("model.") else k
        sd[nk] = v
    return m, sd

def mapping_report(mapping, limit=80):
    """
    Función de diagnóstico que imprime la correspondencia de claves del modelo propio con claves del modelo original
    """
    print(f"Número de entradas del mapping: {len(mapping)}")
    for i, (mk, uk) in enumerate(mapping.items()):
        if i >= limit:
            print(f"... hay ({len(mapping)-limit} más)")
            break
        print(f"{mk:55s} <= {uk}")

def build_safe_mapping_strict(my_sd, ul_sd, block_pairs):
    """
    Construye un diccionario de correspondencias entre el modelo propio y el de Ultralytics.
    Se basa en nombres de prefijos de bloques conocidos y coincidencia de forma

    Args:
      - my_sd: state_dict del modelo propio
      - ul_sd: state_dict del modelo de Ultralytics
      - block_pairs: Lista de bloques equivalentes (prefijo_propio, prefijo_ultralytics)

    Devuelve:
      - mapping: Diccionario de correspondencia de nombres de claves
    """
    mapping = OrderedDict()

    for my_k, my_v in my_sd.items():
        #Exclusión de bloques de head que se entrenarán desde cero
        if my_k.startswith("head."):
            continue

        elegido = None
        for my_pref, ul_pref in block_pairs:
            if my_k.startswith(my_pref):
                elegido = (my_pref, ul_pref)
                break
        if elegido is None:
            continue

        my_pref, ul_pref = elegido
        rel = my_k[len(my_pref):]
        ul_fin = ul_pref + rel

        #Si hubiera bloques C3k2 solo se permite cargar m.* y cv1/cv2 si existe exacto
        if my_pref.startswith("c3k2_") and rel.startswith("cv3."):
            continue

        if my_pref.startswith("c3k2_"):
            if not (rel.startswith("m.") or rel.startswith("cv1.") or rel.startswith("cv2.")):
                continue

        if ul_fin not in ul_sd:
            continue

        if tuple(ul_sd[ul_fin].shape) != tuple(my_v.shape):
            continue

        mapping[my_k] = ul_fin

    return mapping

def apply_mapping(model, ul_sd, mapping, verbose=False):
    """
    Realiza la transferencia física de pesos siguiendo el plan de mapeo
    y actualiza el estado del modelo en memoria.
    """
    my_sd = model.state_dict()
    loaded = 0

    for my_k, ul_k in mapping.items():
        #comprobaciones otra vez
        if my_k not in my_sd or ul_k not in ul_sd:
            continue
        if tuple(my_sd[my_k].shape) != tuple(ul_sd[ul_k].shape):
            continue

        my_sd[my_k] = ul_sd[ul_k]
        loaded += 1
        if verbose:
            print(f"[LOAD] {my_k} <= {ul_k} forma={tuple(my_sd[my_k].shape)}")

    model.load_state_dict(my_sd, strict=False) #el diccionario de pesos estará incompleto respecto a la definición del modelo, por eso strict=False
    return loaded

def initialize_yolo_weights(model):
    """
    Inicia parámetros de la red (Convoluciones, BatchNormalization y Lineales) usando distribuciones estadísticas estándar
    para así asegurar una convergencia estable en las partes no pre-entrenadas.

    Ojo! Se debe llamar antes de apply_mapping, si no se borrarán los pesos preentrenados
    """
    for m in model.modules():

        # Conv2d con Kaiming Normal
        if isinstance(m, torch.nn.Conv2d):
            torch.nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
            if m.bias is not None:
                torch.nn.init.zeros_(m.bias)

        # BatchNorm2d
        elif isinstance(m, torch.nn.BatchNorm2d):
            torch.nn.init.ones_(m.weight)
            torch.nn.init.zeros_(m.bias)
            #Valores específicos de YOLO
            m.eps = 1e-3
            m.momentum = 0.03

        # Linear layers
        elif isinstance(m, torch.nn.Linear):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.01)
            if m.bias is not None:
                torch.nn.init.zeros_(m.bias)

**Función de ensamblaje**

In [19]:
def collate_fn(batch):
    """
    Función de batching de lotes (ensamblaje) para el DataLoader.
    Las imágenes tienen un número variable de boxes que impide el apilado directo, por ello
    se transforma una lista de tuplas (imagen, targets) en dos tensores:
      - imgs: Tensor [Batch_Size, C, H, W] de imágenes
      - targets: Tensor [Total_Boxes_en_batch, 6] (batch_idx, clase, x, y, w, h) de anotaciones
        donde se añade un índice para saber a qué imagen pertenece cada box
    """
    imgs, targets = zip(*batch) #trasponer una linea de tuplas [(img1, tgt1), (img2, tgt2), ...] a (img1, img2, ...), (tgt1, tgt2, ...)
    imgs = torch.stack(imgs, 0) #conviertir la tupla de imágenes en un único tensor
    new_targets = []
    for i, t in enumerate(targets):
        if t.shape[0] > 0: #si la imagen tiene objetos
            # t es [N, 5], pero se quiere [N, 6] -> (batch_idx, cls, x, y, w, h)
            idx = torch.full((t.shape[0], 1), i, dtype=torch.float32)
            new_targets.append(torch.cat((idx, t), 1))

    targets = torch.cat(new_targets, 0) if len(new_targets) > 0 else torch.zeros((0, 6))
    return imgs, targets

**Funciones de entrenamiento y validación**

In [20]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch_idx):
    """
    Ejecución de una época. Itera sobre los lotes, realiza la propagación hacia adelante (forward),
    calcula la pérdida, realiza retropropagación (backward) para actualizar los pesos y devuelve la pérdida promedio.
    """
    model.train()
    running_loss = 0.0

    #Barra de progreso para los batches de una época
    pbar = tqdm(loader, desc=f"   Train", leave=False)

    for imgs, targets in pbar:
        imgs, targets = imgs.to(device), targets.to(device)

        optimizer.zero_grad()
        preds = model(imgs) #Forward

        #Calcular Loss
        loss = criterion(preds, targets)

        #Backward
        loss.backward()
        optimizer.step()

        #Actualizar métricas
        loss_val = loss.item()
        running_loss += loss_val

        #Actualizar la barra con la loss actual
        pbar.set_postfix({'loss': f'{loss_val:.4f}'})

    return running_loss / len(loader)

def compute_precision_recall_f1(boxes_pred, boxes_gt, iou_thres=train_config.iou_thres):
    """
    Cálculo las métricas de clasificación (Precisión, Recall y F1-Score) para una imagen con un umbral de IoU.
    Se utiliza la estrategia de emparejamiento voraz (greedy matching) para asignar predicciones a objetos reales.
    """

    if boxes_gt.numel() == 0:
        #Si no hay GT, definimos precisión=1 si no predice nada, si no 0 (penaliza FP)
        if boxes_pred is None or boxes_pred.numel() == 0:
            return 1.0, 1.0, 1.0
        return 0.0, 1.0, 0.0
    # Si hay GT pero no hay predicciones, el recall es 0
    if boxes_pred is None or boxes_pred.numel() == 0:
        return 0.0, 0.0, 0.0

    #box_iou de torchvision solo acepta 4 columnas, se recorta cualquier columna extra (p.ej class)
    if boxes_pred.size(-1) > 4:
        boxes_pred = boxes_pred[:, :4]
    if boxes_gt.size(-1) > 4:
        boxes_gt = boxes_gt[:, :4]

    ious = torchvision.ops.box_iou(boxes_pred.float(), boxes_gt.float()) # Retorna [N, M]
    matched_gt = set() #Evita que dos predicciones distintas cuenten como acierto para la misma box real
    tp = 0
    #Se itera sobre cada predicción y se asigna la que tenga el mayor IoU
    for i in range(ious.size(0)):
        max_iou, gt_idx = ious[i].max(0)
        if max_iou >= iou_thres and gt_idx.item() not in matched_gt:
            tp += 1
            matched_gt.add(gt_idx.item())

    fp = boxes_pred.size(0) - tp #Falsos positivos
    fn = boxes_gt.size(0) - tp   #Falsos negativos

    #Cálculo de métricas
    precision = tp / (tp + fp + 1e-6) #Epsilon para evitar división por cero
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)

    return float(precision), float(recall), float(f1)


def validate_one_epoch(model, loader, criterion, device, img_size=train_config.img_size):
    """
    Evalúa el modelo en el conjunto de validación.
    Calcula la pérdida agregada y decodifica las predicciones de cada imagen para calcular métricas (P, R, F1).
    """
    model.eval()
    running_loss = 0.0
    recall_list = []
    precision_list = []
    f1_list =[]

    pbar = tqdm(loader, desc="   Valid", leave=False)

    with torch.no_grad():
        for imgs, targets in pbar:
            imgs = imgs.to(device)
            targets = targets.to(device)

            #Inferencia
            preds = model(imgs)
            loss = criterion(preds, targets)

            running_loss += loss.item()

            batch_size = imgs.size(0)

            for i in range(batch_size):
                #GT

                gt_mask = targets[:, 0] == i              #targets es [batch_idx, class, cx, cy, w, h] (6)
                gt_boxes_norm = targets[gt_mask][:, 2:6]  #Nos quedamos con col 2:6 (cx, cy, w, h)

                if gt_boxes_norm.numel() == 0:
                    continue

                #De formato Centro (xywh) a Esquinas (xyxy) y de [0, 1] a píxeles
                gt_boxes = xywh_to_xyxy(gt_boxes_norm) * img_size

                #Extraemos las predicciones de la imagen i
                preds_i = [p[i:i+1] for p in preds]
                boxes_pred = postprocess_single_prediction(
                    preds_i,
                    img_size,
                    conf_thres=train_config.conf_thres,
                    iou_thres=0.65
                )

                prec, rec, f1 = compute_precision_recall_f1(boxes_pred, gt_boxes, iou_thres=train_config.iou_thres)
                precision_list.append(prec)
                recall_list.append(rec)
                f1_list.append(f1)

            #Actualizar barra de progreso
            mean_recall_prog = np.mean(recall_list) if recall_list else 0.0
            pbar.set_postfix({
                'val_loss': f'{loss.item():.4f}',
                'recall': f'{mean_recall_prog:.3f}'
            })

    final_loss = running_loss / len(loader)
    final_recall = float(np.mean(recall_list)) if recall_list else 0.0

    return final_loss, float(np.mean(precision_list)), float(np.mean(recall_list)), float(np.mean(f1_list))


**Código main del entrenamiento y validación**

In [ ]:
VIZ_INTERVAL = 10
SAVE_CKPT = 20
IMG_DIR = train_images_root
OUTPUT_DIR = "data/output"

if __name__ == "__main__":

    set_seed(94)
    train_config = TrainingConfig(variante="yolov8s")

    logger = logs_function(output_dir=OUTPUT_DIR, num_epochs=train_config.num_epochs)

    logger.log_hyperparams(
        folds=1,
        epochs_per_fold=train_config.num_epochs,
        batch_size=train_config.batch_size,
        img_size=train_config.img_size,
        learning_rate=train_config.learning_rate,
        weight_decay=train_config.weight_decay,
        warmup_epochs=train_config.warmup_epochs,
        width=train_config.width,
        depth=train_config.depth,
        conf_thres=train_config.conf_thres,
        iou_thres=train_config.iou_thres,
        device=train_config.device,
    )

    try:

        #Dataset completo
        full_dataset = YOLODataset(IMG_DIR, anotaciones, train_config.img_size)

        #Imagen fija para visualización durante el entrenamiento para hacer un seguimiento
        sample_img_t, sample_targets = None, None
        index = torch.randperm(len(full_dataset)).tolist()

        for i in index:
            img, target = full_dataset[i]
            if target.shape[0] > 0:
                sample_img_t = img.unsqueeze(0)
                sample_targets = target
                logger.write(
                    f"Imagen de muestra para visualización: índice {i}, "
                    f"{len(target)} personas"
                )
                break

        #Split train 80 / validation 20
        train_size = int(0.8 * len(full_dataset))
        val_size = len(full_dataset) - train_size

        train_dataset, val_dataset = random_split(
            full_dataset,
            [train_size, val_size],
            generator=torch.Generator().manual_seed(42)
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=train_config.batch_size,
            shuffle=True,
            collate_fn=collate_fn
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=train_config.batch_size,
            shuffle=False,
            collate_fn=collate_fn
        )

        #Modelo
        model = YOLO_model(num_classes=1, width=train_config.width, depth=train_config.depth, max_ch=train_config.max_ch).to(train_config.device)
        initialize_yolo_weights(model)

        #Descarga de pesos preentrenados
        ul_model, ul_sd = get_ultralytics_sd(train_config.pt_file, device="cpu")

        #Correspondencia de claves de los bloques de pesos (entre el modelo propio y el de Ultralytics)
        block_pairs = [
            ("stem.",   "0."),
            ("conv2.",  "1."),
            ("c2f_2.", "2."),
            ("conv3.",  "3."),
            ("c2f_3.", "4."),
            ("conv4.",  "5."),
            ("c2f_4.", "6."),
            ("conv5.",  "7."),
            ("c2f_5.", "8."),
            ("sppf.",   "9."),
            # No se cargan pesos del Neck ni del Head, difieren del PAN de Ultralytics
        ]

        #Construir el mapping de correspondencia de pesos entre el modelo propio y el de Ultralytics
        my_sd = model.state_dict()
        mapping = build_safe_mapping_strict(model.state_dict(), ul_sd, block_pairs)

        #Carga de pesos para Transfer Learning
        loaded = apply_mapping(model, ul_sd, mapping, verbose=True)
        print(f"[INFO] Pesos cargados: {loaded} tensors")

        #mapping_report(mapping, limit=200)

        #Optimizador
        optimizer = optim.AdamW(
            model.parameters(),
            lr=train_config.learning_rate,
            weight_decay=train_config.weight_decay
        )

        #Configuración del warmup
        warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=train_config.warmup_start, total_iters=train_config.warmup_epochs)

        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=train_config.num_epochs,
            eta_min=1e-5  #Lr mínimo, nunca 0
        )

        #Scheduler compuesto por el warmup + cosine
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[warmup, cosine],
            milestones=[train_config.warmup_epochs]
        )

        #Función de pérdida
        criterion = YOLOLoss(img_size=train_config.img_size)

        training_history = []
        best_metric = -1.0

        print("\nIniciando entrenamiento YOLO")
        print("-" * 60)

        total_start_time = time.time()

        for epoch in range(train_config.num_epochs):

            epoch_start = time.time()

            #training
            train_loss = train_one_epoch(
                model, train_loader, criterion,
                optimizer, train_config.device, epoch
            )

            #validation
            val_loss, val_prec, recall, val_f1 = validate_one_epoch(
                model, val_loader, criterion,
                train_config.device
            )

            epoch_time = time.time() - epoch_start
            current_lr = optimizer.param_groups[0]['lr']

            #Visualización de imagen de ejemplo para seguir el progreso del entrenamiento
            if (epoch % VIZ_INTERVAL == 0 or epoch == train_config.num_epochs - 1) and sample_img_t is not None:
                plot_progress(model, sample_img_t, sample_targets, epoch, fold=1, device=train_config.device, save_dir=OUTPUT_DIR)

            #Guardado del checkpoint de pesos para el mejor F1
            saved = "no"
            if val_f1 > best_metric:
                best_metric = val_f1
                torch.save(model.state_dict(), f"{OUTPUT_DIR}/yolo_nano_best.pth")
                saved = "yes"

            #Guardado del checkpoint del modelo completo cada SAVE_CKPT épocas
            if ((epoch+1) % SAVE_CKPT == 0):
              ckpt_path = f"{OUTPUT_DIR}/yolo_epoch{epoch+1:03d}.pth"
              torch.save(model.state_dict(), ckpt_path)
              ckpt = {
                  "epoch": epoch+1,
                  "model_state_dict": model.state_dict(),
                  "optimizer_state_dict": optimizer.state_dict(),
                  "scheduler_state_dict": scheduler.state_dict(),
                  "best_metric": best_metric,
                  "config": train_config.__dict__ if hasattr(train_config, "__dict__") else None
              }
              torch.save(ckpt, f"{OUTPUT_DIR}/yolo_ckpt_epoch{epoch+1:03d}.pt")

            #Logs por pantalla
            print(
                f"Epoch {epoch+1}/{train_config.num_epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Prec: {val_prec:.4f} | "
                f"Recall: {recall:.4f} | "
                f"F1: {val_f1:.4f} | "
                f"LR: {current_lr:.1e} | "
                f"Time: {epoch_time:.2f}s | "
                f"Saved: {saved}"
            )


            #Recogida de logs
            logger.log_results(
                epoch=epoch+1,
                train_loss=train_loss,
                val_loss=val_loss,
                precision=val_prec,
                recall=recall,
                f1=val_f1,
                lr=current_lr,
                epoch_time=round(epoch_time, 2),
                saved=saved
            )

            #Guardar en historial
            training_history.append({
                "Epoch": epoch + 1,
                "LR": current_lr,
                "Epoch_Time_Sec": round(epoch_time, 2),
                "Train_Loss": train_loss,
                "Val_Loss": val_loss,
                "Precision": val_prec,
                "Recall": recall,
                "F1": val_f1,
                "Saved": saved
            })

            scheduler.step()

        total_time = time.time() - total_start_time
        logger.write(f"Tiempo total de entrenamiento: {round(total_time, 2)} segundos")

    except Exception as e:
        logger.log_error(str(e))
        raise e

    print("\nEntrenamiento completado.")

    ############################
    # EXPORTAR MÉTRICAS A EXCEL
    ############################
    print("[INFO] Generando Excel de métricas...")

    df = pd.DataFrame(training_history)

    #Configuración fija
    df["Batch_Size"] = train_config.batch_size
    df["Image_Size"] = train_config.img_size
    df["Learning_Rate"] = train_config.learning_rate
    df["Weight_Decay"] = train_config.weight_decay
    df["Optimizer"] = "AdamW"

    excel_path = f"{OUTPUT_DIR}/metricas_yolo_train_val.xlsx"

    try:
        df.to_excel(excel_path, index=False)
        print(f"[ÉXITO] Métricas guardadas en: {excel_path}")
    except Exception as e:
        csv_path = excel_path.replace(".xlsx", ".csv")
        df.to_csv(csv_path, index=False)
        print(f"[WARNING] Excel falló, guardado como CSV: {csv_path}")


Imagen de muestra para visualización: índice 1403, 12 personas
[LOAD] stem.conv.weight <= 0.conv.weight forma=(32, 3, 3, 3)
[LOAD] stem.bn.weight <= 0.bn.weight forma=(32,)
[LOAD] stem.bn.bias <= 0.bn.bias forma=(32,)
[LOAD] stem.bn.running_mean <= 0.bn.running_mean forma=(32,)
[LOAD] stem.bn.running_var <= 0.bn.running_var forma=(32,)
[LOAD] stem.bn.num_batches_tracked <= 0.bn.num_batches_tracked forma=()
[LOAD] conv2.conv.weight <= 1.conv.weight forma=(64, 32, 3, 3)
[LOAD] conv2.bn.weight <= 1.bn.weight forma=(64,)
[LOAD] conv2.bn.bias <= 1.bn.bias forma=(64,)
[LOAD] conv2.bn.running_mean <= 1.bn.running_mean forma=(64,)
[LOAD] conv2.bn.running_var <= 1.bn.running_var forma=(64,)
[LOAD] conv2.bn.num_batches_tracked <= 1.bn.num_batches_tracked forma=()
[LOAD] c2f_2.cv1.conv.weight <= 2.cv1.conv.weight forma=(64, 64, 1, 1)
[LOAD] c2f_2.cv1.bn.weight <= 2.cv1.bn.weight forma=(64,)
[LOAD] c2f_2.cv1.bn.bias <= 2.cv1.bn.bias forma=(64,)
[LOAD] c2f_2.cv1.bn.running_mean <= 2.cv1.bn.running_

   Train:   2%|▏         | 3/128 [00:04<02:53,  1.38s/it, loss=23.6256]

In [21]:
import gc
# Liberación de memoria
gc.collect()
torch.cuda.empty_cache()